## Function to fetch data:

In [15]:
import os
import tarfile
import urllib

DOWNLOAD_ROOT = "https://raw.githubusercontent.com/ageron/handson-ml2/master/"
HOUSING_PATH = os.path.join("datasets", "housing")
HOUSING_URL = DOWNLOAD_ROOT + "datasets/housing/housing.tgz"

def fetch_housing_data(housing_url=HOUSING_URL, housing_path=HOUSING_PATH):
    os.makedirs(housing_path, exist_ok=True)
    tgz_path = os.path.join(housing_path, "housing.tgz")
    urllib.request.urlretrieve(housing_url, tgz_path)
    housing_tgz = tarfile.open(tgz_path)
    housing_tgz.extractall(path=housing_path)
    housing_tgz.close()

## Function to load data into notebook:

In [16]:
import pandas as pd

def load_housing_data(housing_path=HOUSING_PATH):
    csv_path = os.path.join(housing_path, "housing.csv")
    return pd.read_csv(csv_path)

## Data overview and visualizations:

In [ ]:
housing.describe()                          # 只显示numerical columns，null values会被忽略
housing['ocean_proximity'].value_counts()         # 通常用于categorical data，统计dataframe中某一列每个数的出现次数

corr_matrix = housing.corr()
corr_matrix['median_house_value'].sort_values(ascending=True)    # 计算attributes之间的correlation，并将每个attribute与median house value的相关性按降序排列。corr只考虑线性关系的强度

import matplotlib.pyplot as plt             # 对选中的一列数值型变量画histogram，可以找出哪些变量被缩放/压制过，，以及label的区间是多少(直接决定模型能预测的范围)，不同变量的数值之间差异是否需要后面进行调整，每个数值型变量的分布情况
plt.figure(figsize = (8, 6))
plt.hist(df['col_name'], bins = )
plt.title('')
plt.xlabel('')
plt.ylabel('')
plt.show()

plt.scatter(housing['longitude'], housing['latitude'], alpha=0.1, s=housing['population']/100, c=housing['median_house_value'], cmap=plt.get_cmap('jet'))                # 对某两组数值型变量画散点图，alpha是点的透明度可以显示哪个区域数据密度更高，s是点的大小，c是点的数值，cmap是点的数值映射的颜色
plt.title('')
plt.xlabel('')
plt.ylable('')
plt.colorbar()                              # 显示颜色条
plt.show()

## Train/Test split:

In [ ]:
from zlib import crc32
import numpy as np

def test_set_check(identifier, test_ratio):
    return crc32(np.int64(identifier)) & 0xffffffff < test_ratio * 2**32            # crc32(np.int64(identifier))对输入的ID输出一个固定且单一的hash value，& 0xffffffff可以把hash value控制在[0, 2*32]这个区间
                                                                                    # 如果输出的hash value小于test_ratio * 2**32,那么函数输出true，这个观测应该被放入test set

def split_train_test_by_id(data, test_ratio, id_column):
    ids = data[id_column]                                                           # 取出id列
    in_test_set = ids.apply(lambda id_: test_set_check(id_, test_ratio))            # 对每一个id，都运行一次test_set_check()，返回一个true false true true....的vector叫做in_test_set
    return data.loc[~in_test_set], data.loc[in_test_set]                            # data.loc[in_test_set]取出in_test_set == True的行，data.loc[~in_test_set]取出in_test_set == False的行

housing_with_id = housing.reset_index()                                             # 增加index列，其中保留的是原始索引。但问题是如果后来在housing df间删除了一列，那么整个housing df的index都变了，原来的test row可能会变成train row，而核心目标是train/test需要永远分开
train_set, test_set = split_train_test_by_id(housing_with_id, 0.2, "index")

housing_with_id["id"] = housing["longitude"] * 1000 + housing["latitude"]           # 另外一种方法：用恒定不变的信息例如房子的经纬度做id，这样每个房子的id是不会因为增加/删除某观测而改变的
train_set, test_set = split_train_test_by_id(housing_with_id, 0.2, "id")


from sklearn.model_selection import train_test_split                                # sklearn里有已经做好的train/test split function。需要设置random state保证两个set内容固定。这个function的好处是可以同时划分多个数据集：
train_set, test_set = train_test_split(housing, test_size=0.2, random_state=42)     # 如果X=features df, Y=labels df, X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=42)可以保证X_train, Y_train对应的是同一批行；X_test, Y_test对应的是同一批行


from sklearn.model_selection import StratifiedShuffleSplit
housing['income_cat'] = pd.cut(housing['median_income'],                            # 当dataframe足够大的时候，随机选择train/test没有问题。当dataframe不够大的时候，随机选择的train/test需要有代表性。因此需要根据重点attributes进行分层。按比例每层筛选
                               bins=[0,1.5,3,4.5,6,np.inf],                         # 根据median_income给每行分类并作标记
                               labels=[1,2,3,4,5])
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)          # n_splits=1:只剩成一组train index和test index
for train_index, test_index in split.split(housing, housing["income_cat"]):         # 将housing按照income_cat这一列的比例划分
    strat_train_set = housing.loc[train_index]                                      # loc: 根据标签选择，iloc: 根据行列数选择
    strat_test_set = housing.loc[test_index]
strat_train_set = strat_train_set.drop('income_cat', axis=1)
strat_test_set = strat_test_set.drop('income_cat', axis=1)

housing = strat_tain_set.drop(['median_house_value', axis=1])                       # 创建没有label的train set
housing_labels = strat_train_set['median_house_value'].copy()                       # 创建label，label的entries和train set observations依然是对应着的，训练模型之后还是可以和label正常比对的

## Data cleaning steps：

In [ ]:
# 一些原始变量本身意义不够强，比如total_rooms;population;households，需要做一些改进：
housing['rooms_per_household'] = housing['total_rooms']/housing['households']           # 对于一个district，total rooms的意义不够rooms per household（每户平均多少rooms）意义大
housing['bedrooms_per_room'] = housing['total_bedrooms']/housing['total_rooms']         # bedrooms per room小代表除了卧室意外会有更多其余空间比如厨房客厅等等

# na values:
housing.dropna(subset=['total_bedrooms'])                     # 删除total bedrooms是空值的行
housing.drop('total_bedrooms', axis=1)                        # 删除整个total bedrooms这一列

from sklearn.impute import SimpleImputer 
housing_num = housing.drop('ocean_proximity', axis = 1)
imputer = SimpleImputer(strategy = 'median')                  # 定义一个补充缺失值的imputer工具，每一列的缺失用这一列的median补充, 注意这里只能将imputer应用在数值型数据上
imputer.fit(housing_num)                                      # 用housing_num创建这个imputer工具，他会把每一列的median计算出来并做成一个np array，每一列的median可以用imputer.statistics_调用
X = imputer.transform(housing_num)                            # 用imputer补充housing_num的所有缺失值，但输出的是一个np array
housing_tr = pd.DataFrame(X, columns=housing_num.columns, index=housing_num.index) # 补充缺失值之后转换回dataframe，填充旧列名和旧的行index

# categorical attributes:
from sklearn.preprocessing import OneHotEncoder
housing_cat = housing[['ocean_proximity']]                    # ocean proximity: near ocean; <1h ocean; in land; island; near bay
cat_encoder = OneHotEncoder()                                 # categorical variable有几种类别，one hot encoder就会创建几列。对于每一个观测，只有一列是1，其余是0
housing_cat_1hot = cat_encoder.fit_transform(housing_cat)     # 输出是一个和housing行数一样，一共五列的sparse matrix

## Data cleaning pipeline:

In [ ]:
housing['rooms_per_household'] = housing['total_rooms']/housing['households']          
housing['bedrooms_per_room'] = housing['total_bedrooms']/housing['total_rooms']
housing_num = housing.drop('ocean_proximity', axis=1)
num_attributes = list(housing_num)                            # list(df)会输出一个df所有列名的list
cat_attributes = ['ocean_proximity']

# write a pipeline for data cleaning:                         
# Pipeline([
    # ('step1_name', transformer 1),
    # ...
    # ('stepn_name', transformer n),
# ])

# ColumnTransformer([
    # ('step1_name', transformer 1, columns to apply transformer 1 to),
    # ...
    # ('stepn_name', transformer 2, columns to apply transformer n to),
])

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

num_pipeline = Pipeline([                                     # 先写一个对于numerical attributes的pipeline
    ('imputer', SimpleImputer(strategy='median')),            # 每一个transformer都是先fit()再transform()
    ('std_scaler', StandardScaler()),
])

full_pipeline = ColumnTransformer([                           # 可以对不同列使用不同的transformer
    ('num', num_pipeline, num_attributes),
    ('cat', OneHotEncoder(), cat_attributes),
])

housing_prepared = full_pipeline.fit_transform(housing)       # 最后会输出一个matrix



## Train a model (multiple linear regression model for example):

In [ ]:
from sklearn.linear_model import LinearRegression
linear_model = LinearRegression()
linear_model.fit(housing_prepared, housing_labels)
housing_predictions = linear_model.transform(housing_prepared)

from sklearn.metrics import mean_squared_error                  # 计算模型RMSE，如果RMSE比较大，代表underfit
mse = mean_squared_error(housing_labels, housing_predictions)
rmse = np.sqrt(mse)

## K-fold Cross Validation:

In [ ]:
from sklearn.model_selection import cross_val_score 
# cross_val_score(name of the model for evaluation, cleaned and processed matrix, labels, scoreing = '' (error measuring metric), cv = (number of folds))

scores = cross_val_score(
    linear_model,
    housing_prepared,
    housing_labels,
    cv = 10,
    scoring = 'neg_mean_squared_error'                            # sklearn的设计原则是分数越大越好，但对于MRSE实际是越小越好。所以说对于这种metric需要用户手动计算负值再转回正值。如这里的代码
)
linear_model_rmse_scores = np.sqrt(-scores)                       # 会输出一个长度为k的array
linear_model_rmse_scores
linear_model_rmse_scores.mean()                                   # 如果rmse_scores.mean() > rmse，代表模型存在overfitting。解决方案是简化模型
linear_model_rmse_scores.std()

## Save/delete models/results for later use:

In [ ]:
import joblib                                                           # 保存的pkl是可以跨notebook调取的
joblib.dump(full_pipeline, 'full_pipeline.pkl')                         # 保存full pipeline，之后不用重新写
joblib.dump(linear_model, 'linear_model.pkl')                           # 保存linear model（截距，系数）
joblib.dump(linear_model_rmse_scores, 'linear_model_rmse_scores.pkl')   # 保存cross validation的结果，之后不需要重新运行程序

linear_model_loaded = joblib.load("linear_model.pkl")                   # 调取之前保存好的模型

import os 
os.remove('linear_model.pkl')                                           # 删除保存好的模型 